In [17]:
!pip install duckduckgo-search

  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
Using cached click-8.5.0-py3-none-any.whl (125 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 13.7 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 13.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [duckduckgo-search]


In [11]:
import json
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-120b"



In [12]:
ROADMAP_SCHEMA = {
    "type": "object",
    "properties": {
        "idea_title": {"type": "string"},
        "phases": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "phase_name": {"type": "string"},
                    "steps": {
                        "type": "array",
                        "items": {"type": "string"},
                    },
                    "milestone": {"type": "string"},
                    "skills_required": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "skill_name": {"type": "string"},
                                "resources": {
                                    "type": "array",
                                    "items": {
                                        "type": "object",
                                        "properties": {
                                            "title": {"type": "string"},
                                            "url": {"type": "string"},
                                        },
                                        "required": ["title", "url"],
                                        "additionalProperties": False,
                                    },
                                },
                            },
                            "required": ["skill_name", "resources"],
                            "additionalProperties": False,
                        },
                    },
                },
                "required": [
                    "phase_name",
                    "steps",
                    "milestone",
                    "skills_required",
                ],
                "additionalProperties": False,
            },
        },
    },
    "required": ["idea_title", "phases"],
    "additionalProperties": False,
}

SYSTEM_PROMPT = """You are an expert project mentor. Given a project or startup idea,
break it into 3-5 sequential phases. For each phase, give:
- concrete implementation steps (not vague advice)
- one clear milestone that marks the phase as complete
- the specific skills/frameworks needed for that phase.

CRITICAL INSTRUCTION: You MUST use your browser search tool to find 1-2 genuinely useful, accurate, and up-to-date resource links (docs, well-known tutorials, or courses) for each skill. Do not hallucinate or guess URLs.

Be specific to the idea given. Do not pad with generic advice."""

In [18]:
import json
from duckduckgo_search import DDGS

def web_search(query: str, max_results: int = 3) -> str:
    """Executes a DuckDuckGo search and returns formatted results."""
    try:
        # DDGS().text handles the actual search query
        results = DDGS().text(query, max_results=max_results)
        if not results:
            return "No results found."
        
        # Format the results into a string the LLM can easily read
        formatted_results = []
        for r in results:
            formatted_results.append(f"Title: {r['title']}\nURL: {r['href']}\nSnippet: {r['body']}")
        
        return "\n\n".join(formatted_results)
    except Exception as e:
        return f"Search error: {str(e)}"

# Define the schema Groq needs to understand our tool
tools = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for up-to-date documentation, tutorials, or courses.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The exact search query (e.g., 'React official documentation' or 'PostgreSQL beginner tutorial')"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

In [15]:
def generate_roadmap(idea_text: str) -> dict:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": idea_text},
        ],
        
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "roadmap",
                "schema": ROADMAP_SCHEMA,
                "strict": True,
            },
        },
    )
    return json.loads(response.choices[0].message.content)

In [16]:
idea = "A web app that helps users track their daily habits and provides insights on their progress."
roadmap = generate_roadmap(idea)
print(roadmap)

{'idea_title': 'Daily Habit Tracker Web App', 'phases': [{'phase_name': 'Research & Design', 'steps': ['Define core habit-tracking features and success metrics.', 'Create user personas and journey maps.', 'Sketch low‑fidelity wireframes for dashboard, habit entry, and analytics view.', 'Draft the data model (users, habits, daily entries).', 'Review mockups with potential users for feedback.'], 'milestone': 'Approved UI mockups and finalized relational data schema.', 'skills_required': [{'skill_name': 'Wireframing & UI Design', 'resources': [{'title': 'Figma Learn Design Resources', 'url': 'https://www.figma.com/resources/learn-design/'}]}, {'skill_name': 'Relational Database Design (PostgreSQL)', 'resources': [{'title': 'PostgreSQL Official Tutorial', 'url': 'https://www.postgresql.org/docs/current/tutorial.html'}]}]}, {'phase_name': 'Frontend MVP', 'steps': ['Initialize a Git repository and create a React project with Vite.', 'Add Tailwind CSS for rapid styling.', 'Implement authentic

In [23]:
# 1. Scoped prompt focusing only on core technical frameworks
RESEARCHER_PROMPT = """You are an expert technical mentor. Given a project or startup idea:
1. Break it into 3-5 sequential phases.
2. For each phase, list implementation steps, milestone, and required skills.
3. SELECTIVITY: Only provide resource links for the TOP 2-4 core technical frameworks or libraries across the entire project (e.g., React, PostgreSQL, Docker). Do NOT search for soft skills, basic concepts, or generic topics (e.g., "User Research", "Git basics").

CRITICAL SEARCH RULE:
- You are allowed ONE round of searches.
- Submit all necessary queries in parallel at the same time using `web_search`.
- After receiving the search results, immediately generate your full response."""


def run_phase_1_researcher(idea_text: str):
    messages = [
        {"role": "system", "content": RESEARCHER_PROMPT},
        {"role": "user", "content": idea_text}
    ]
    
    print("🧠 Step 1: Generating roadmap & deciding top technical searches...")
    
    # First call: Model determines roadmap and fires off search queries in parallel
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto",
        parallel_tool_calls=True
    )
    
    response_message = response.choices[0].message
    messages.append(response_message)
    
    # Step 2: If the model called searches, execute them all in one batch
    if response_message.tool_calls:
        print(f"🔄 Executing {len(response_message.tool_calls)} targeted searches...")
        
        for tool_call in response_message.tool_calls:
            if tool_call.function.name == "web_search":
                args = json.loads(tool_call.function.arguments)
                query = args.get("query", "")
                
                print(f"  ↳ 🔍 Searching: '{query}'")
                search_results = web_search(query)
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": "web_search",
                    "content": search_results
                })
        
        # Step 3: Hard Stop / Final Synthesis
        # We append a brief system directive instructing the model to synthesize immediately
        messages.append({
            "role": "user",
            "content": "All requested searches are complete. Now synthesize everything into the final detailed text summary with the verified links."
        })
        
        print("✍️ Step 2: Synthesizing final roadmap summary...")
        final_response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            # tool_choice="none" ensures it cannot attempt another search round
            tools=tools,
            tool_choice="none" 
        )
        return final_response.choices[0].message.content
        
    else:
        print("ℹ️ Model answered directly without search.")
        return response_message.content

# Run it
researched_text = run_phase_1_researcher(idea)
print("\n=== PHASE 1 OUTPUT ===\n")
print(researched_text)

🧠 Step 1: Generating roadmap & deciding top technical searches...
🔄 Executing 1 targeted searches...
  ↳ 🔍 Searching: 'React official documentation'


/var/folders/vk/4d6vvqtd6k39w3345n5kc3_80000gn/T/ipykernel_30665/308555504.py:8: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  results = DDGS().text(query, max_results=max_results)


✍️ Step 2: Synthesizing final roadmap summary...

=== PHASE 1 OUTPUT ===

Below is a **step‑by‑step roadmap** for building a habit‑tracking web app, broken into **four logical phases**. For each phase you’ll see the concrete implementation steps, the milestone that tells you when the phase is “done”, and the core technical skills you need to have (or acquire).  

Only the **four most critical frameworks/libraries** that the whole project will rely on are linked – React, Express (Node.js), PostgreSQL, and Docker. These links point to the official docs and high‑quality “getting‑started” guides, so you can dive straight into coding without getting lost in peripheral material.

---

## 📅 Phase Overview

| Phase | Focus | Typical Duration |
|------|-------|------------------|
| **1️⃣ UI/UX Foundations & Front‑end Scaffold** | Design, React component architecture, state‑management, UI kit | 2‑3 weeks |
| **2️⃣ Backend API & Data Model** | Express server, REST endpoints, habit & user schema, 

In [24]:
FORMATTER_PROMPT = """You are a strict data formatting assistant. 
Your only job is to take the provided text and map it exactly into the required JSON schema.

RULES:
1. Do not invent, guess, or hallucinate any information.
2. Only include skills and resource links that are explicitly mentioned in the provided text.
3. If a phase doesn't mention specific resources for a skill, leave the resources array empty.
4. Ensure the output strictly validates against the requested JSON schema."""

def run_phase_2_formatter(researched_data: str) -> dict:
    print("🧱 Formatting data into structured JSON...")
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": FORMATTER_PROMPT},
            {"role": "user", "content": researched_data},
        ],
        # Notice: No `tools` parameter here!
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "roadmap",
                "schema": ROADMAP_SCHEMA,
                "strict": True,
            },
        },
    )
    
    # Parse the string response into a Python dictionary
    return json.loads(response.choices[0].message.content)

# Run the formatter using the text from Phase 1
final_json_roadmap = run_phase_2_formatter(researched_text)

print("\n=== FINAL PHASE 2 JSON OUTPUT ===\n")
print(json.dumps(final_json_roadmap, indent=2))


🧱 Formatting data into structured JSON...

=== FINAL PHASE 2 JSON OUTPUT ===

{
  "idea_title": "Habit-Tracking Web App",
  "phases": [
    {
      "phase_name": "UI/UX Foundations & Front\u2011end Scaffold",
      "steps": [
        "Define user flows (sign\u2011up/login \u2192 habit list \u2192 habit detail \u2192 insights dashboard). Sketch low\u2011fidelity wireframes (Figma/pen\u2011and\u2011paper \u2013 no need for a link).",
        "Set up React project with Vite (fast dev server) or Create\u2011React\u2011App.",
        "Install a UI component library (e.g., Radix UI or MUI) for consistent styling.",
        "Build core pages as functional components: AuthPage (login/register), HabitsPage (list + add habit), HabitDetail (daily check\u2011off), InsightsPage (charts).",
        "Add state\u2011management (React Context + useReducer or Redux Toolkit if you anticipate complex flows).",
        "Integrate React Router for navigation and protect routes with a simple auth guard.",
  

In [25]:
with open("mock_roadmap.json", "w") as f:
    json.dump(final_json_roadmap, f, indent=2)